# UCLA Starbucks Data Challenge

In [ ]:
# Install required packages
!pip install google-generativeai pandas numpy scikit-learn tqdm -q

In [1]:
# Imports
import pandas as pd
import numpy as np
import json
import time
import requests
import os
import re
import ast
from typing import Optional, Dict, List, Any
import google.generativeai as genai
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

print("All imports successful!")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


All imports successful!


In [ ]:
# Configure your Gemini API Key
# Get your key at: https://makersuite.google.com/app/apikey

GEMINI_API_KEY = "" 


genai.configure(api_key=GEMINI_API_KEY)
print("Gemini API configured!")

Gemini API configured!


## Load Data

In [3]:
# Load the three CSV files
# Update these paths based on where you downloaded the files

products = pd.read_csv("products.csv")
train_queries = pd.read_csv("queries_train.csv")
test_queries = pd.read_csv("queries_test.csv")

print(f"Products: {len(products)} items")
print(f"Training queries: {len(train_queries)}")
print(f"Test queries: {len(test_queries)}")

Products: 115 items
Training queries: 100
Test queries: 100


In [4]:
# Explore the products data
products.head(10)

,product_id,name,category,subcategory,temperature,caffeine_mg,calories,sugar_g,protein_g,contains_dairy,contains_nuts,contains_gluten,is_vegan,description,price
0,ESP_001,Caffè Americano,espresso,americano,hot,225,15,0,1,False,False,False,True,Espresso shots topped with hot water for a lig...,4.45
1,ESP_002,Blonde Caffè Americano,espresso,americano,hot,255,15,0,1,False,False,False,True,"Blonde espresso with hot water for a smooth, s...",4.45
2,ESP_003,Caffè Latte,espresso,latte,hot,150,190,18,13,True,False,False,False,Rich espresso balanced with steamed milk and a...,5.45
3,ESP_004,Vanilla Latte,espresso,latte,hot,150,250,35,12,True,False,False,False,Espresso with vanilla syrup and steamed milk,5.95
4,ESP_005,Caramel Macchiato,espresso,macchiato,hot,150,250,33,10,True,False,False,False,Vanilla-flavored drink marked with espresso an...,5.95
5,ESP_006,Caffè Mocha,espresso,mocha,hot,175,360,35,14,True,False,False,False,"Espresso with mocha sauce, steamed milk, and w...",5.75
6,ESP_007,White Chocolate Mocha,espresso,mocha,hot,150,420,53,14,True,False,False,False,"Espresso with white chocolate sauce, steamed m...",5.95
7,ESP_008,Cappuccino,espresso,cappuccino,hot,150,140,10,10,True,False,False,False,Dark espresso with a deep layer of foamed milk,4.95
8,ESP_009,Flat White,espresso,flat_white,hot,195,170,13,11,True,False,False,False,Ristretto shots with steamed whole milk,5.25
9,ESP_010,Honey Almondmilk Flat White,espresso,flat_white,hot,225,170,24,4,False,True,False,True,Blonde espresso with almond milk and honey,5.75


In [5]:
# Check product categories
print("Categories:", products['category'].unique())
print("\nTemperatures:", products['temperature'].unique())
print("\nCaffeine range:", products['caffeine_mg'].min(), "-", products['caffeine_mg'].max())

Categories: ['espresso' 'cold_brew' 'frappuccino' 'other' 'tea' 'refresher' 'brewed']

Temperatures: ['hot' 'iced' 'blended']

Caffeine range: 0 - 360


In [6]:
# Look at sample training queries
train_queries[['query_id', 'query_text', 'relevant_products']].head(5)

,query_id,query_text,relevant_products
0,TRAIN_001,"need a pick me up, something like just black c...","['BRW_002', 'BRW_001', 'BRW_005', 'BRW_003']"
1,TRAIN_002,May I get a chai or tea that's no regular milk...,"['TEA_005', 'ICT_004', 'TEA_008', 'TEA_007', '..."
2,TRAIN_003,hit me with basic brewed coffee that's under 4...,"['BRW_002', 'BRW_001', 'BRW_004', 'BRW_003', '..."
3,TRAIN_004,i want some tea that's less than 200 calories ...,"['TEA_005', 'ICT_004', 'TEA_008', 'TEA_007', '..."
4,TRAIN_005,could i get a cold brew that's under 100 cal a...,"['CBR_001', 'CBR_012', 'CBR_002', 'CBR_011', '..."


In [7]:
# Check available models
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-lite-preview-02-05
models/gemini-2.0-flash-lite-preview
models/gemini-exp-1206
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-preview-09-2025
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-p

## Stage 1: Constraint Extraction (LLM)

Use Gemini to extract structured constraints from natural language queries.

In [8]:
# The prompt template for constraint extraction
CONSTRAINT_EXTRACTION_PROMPT = """You are a Starbucks order assistant. Extract structured constraints from customer queries.

VALID VALUES:
- category: brewed, cold_brew, espresso, frappuccino, refresher, tea (pick the MOST specific match)
- temperature: hot, iced, blended
- caffeine_level: none, low, medium, high
- max_calories, max_sugar, max_price: numbers only (no units)
- dairy_free, vegan: true or false

CATEGORY MAPPING HINTS:
- "plain coffee", "regular coffee", "drip coffee", "house coffee", "black coffee" → brewed
- "cold brew", "nitro" → cold_brew
- "espresso", "latte", "americano", "cappuccino", "macchiato", "mocha" → espresso
- "frappuccino", "frozen", "blended" → frappuccino
- "refresher", "pink drink", "dragon drink", "fruity" → refresher
- "tea", "chai", "matcha", "herbal", "green tea" → tea

TEMPERATURE MAPPING:
- "hot", "warm", "steaming" → hot
- "iced", "cold", "chilled", "on ice" → iced
- "frozen", "blended" → blended

CAFFEINE MAPPING:
- "decaf", "caffeine free", "no caffeine", "without caffeine" → none
- "low caffeine", "mild", "not too much caffeine" → low
- "regular caffeine", "medium caffeine", "normal" → medium
- "high caffeine", "strong", "extra caffeine", "lots of caffeine", "need the caffeine", "pick me up" → high

DIETARY MAPPING:
- "no dairy", "dairy free", "non-dairy", "without milk", "no milk", "no regular milk", "oat milk or something dairy free" → dairy_free: true
- "vegan", "plant based", "no animal stuff", "no animal products" → vegan: true

Return ONLY a valid JSON object with these exact keys (use null for unspecified):
{{
  "category": string or null,
  "temperature": string or null,
  "max_calories": number or null,
  "max_sugar": number or null,
  "max_price": number or null,
  "dairy_free": boolean or null,
  "vegan": boolean or null,
  "caffeine_level": string or null
}}

Customer query: "{query}"

JSON output:"""

In [9]:
class ConstraintExtractor:
    """Stage 1: Extract structured constraints from natural language queries."""

    def __init__(self):
        self.model = genai.GenerativeModel("gemini-2.0-flash-lite")

    def extract(self, query: str) -> Dict[str, Any]:
        """Extract constraints from a single query."""
        prompt = CONSTRAINT_EXTRACTION_PROMPT.format(query=query)

        try:
            response = self.model.generate_content(
                prompt,
                generation_config=genai.GenerationConfig(
                    temperature=0.0,
                    max_output_tokens=500,
                )
            )

            response_text = response.text.strip()

            # Clean up markdown code blocks if present
            if response_text.startswith("```"):
                response_text = re.sub(r'^```json?\s*', '', response_text)
                response_text = re.sub(r'\s*```$', '', response_text)

            constraints = json.loads(response_text)
            if constraints is None:
                return self._empty_constraints()
            return constraints

        except Exception as e:
            print(f"Error: {e}")
            return self._empty_constraints()

    def _empty_constraints(self) -> Dict[str, Any]:
        return {
            "category": None, "temperature": None,
            "max_calories": None, "max_sugar": None, "max_price": None,
            "dairy_free": None, "vegan": None, "caffeine_level": None
        }

# Initialize the extractor
extractor = ConstraintExtractor()
print("Constraint extractor ready!")

Constraint extractor ready!


In [10]:
# Test constraint extraction on a sample query
test_query = "I want something sweet and cold but I'm trying to avoid dairy"
constraints = extractor.extract(test_query)

print(f"Query: {test_query}")
print(f"\nExtracted constraints:")
print(json.dumps(constraints, indent=2))

Query: I want something sweet and cold but I'm trying to avoid dairy

Extracted constraints:
{
  "category": null,
  "temperature": "iced",
  "max_calories": null,
  "max_sugar": null,
  "max_price": null,
  "dairy_free": true,
  "vegan": null,
  "caffeine_level": null
}


In [11]:
# Test on a few more training queries
for i in range(3):
    row = train_queries.iloc[i]
    print(f"\n{'='*60}")
    print(f"Query: {row['query_text']}")

    constraints = extractor.extract(row['query_text'])
    print(f"Extracted: {constraints}")

    # Show expected category from training data
    print(f"Expected category: {row['constraint_category']}")
    time.sleep(4)  # Avoid rate limits


Query: need a pick me up, something like just black coffee that's need the caffeine and under 45 grams of sugar
Extracted: {'category': 'brewed', 'temperature': None, 'max_calories': None, 'max_sugar': 45, 'max_price': None, 'dairy_free': None, 'vegan': None, 'caffeine_level': 'high'}
Expected category: brewed

Query: May I get a chai or tea that's no regular milk and under 250 cal?
Extracted: {'category': 'tea', 'temperature': None, 'max_calories': 250, 'max_sugar': None, 'max_price': None, 'dairy_free': True, 'vegan': None, 'caffeine_level': None}
Expected category: tea

Query: hit me with basic brewed coffee that's under 45 grams of sugar and cheaper than $5.0
Extracted: {'category': 'brewed', 'temperature': None, 'max_calories': None, 'max_sugar': 45, 'max_price': 5.0, 'dairy_free': None, 'vegan': None, 'caffeine_level': None}
Expected category: brewed


## Stage 2: Product Filtering

Filter the product catalog based on extracted constraints.

In [12]:
class ProductFilter:
    """Stage 2: Filter products based on extracted constraints."""

    CAFFEINE_RANGES = {
        "none": (0, 0),
        "low": (1, 50),
        "medium": (51, 175),
        "high": (176, 500)
    }

    def __init__(self, products_df: pd.DataFrame):
        self.products = products_df.copy()

    def filter(self, constraints: Dict[str, Any]) -> pd.DataFrame:
        """Filter products based on constraints."""
        filtered = self.products.copy()

        # Category filter
        if constraints.get("category"):
            filtered = filtered[filtered["category"] == constraints["category"]]

        # Temperature filter
        if constraints.get("temperature"):
            filtered = filtered[filtered["temperature"] == constraints["temperature"]]

        # Max calories filter
        if constraints.get("max_calories") is not None:
            filtered = filtered[filtered["calories"] <= constraints["max_calories"]]

        # Max sugar filter
        if constraints.get("max_sugar") is not None:
            filtered = filtered[filtered["sugar_g"] <= constraints["max_sugar"]]

        # Max price filter
        if constraints.get("max_price") is not None:
            filtered = filtered[filtered["price"] <= constraints["max_price"]]

        # Dairy-free filter
        if constraints.get("dairy_free") is True:
            filtered = filtered[filtered["contains_dairy"] == False]

        # Vegan filter
        if constraints.get("vegan") is True:
            filtered = filtered[filtered["is_vegan"] == True]

        # Caffeine level filter
        if constraints.get("caffeine_level"):
            level = constraints["caffeine_level"]
            if level in self.CAFFEINE_RANGES:
                min_caff, max_caff = self.CAFFEINE_RANGES[level]
                filtered = filtered[
                    (filtered["caffeine_mg"] >= min_caff) &
                    (filtered["caffeine_mg"] <= max_caff)
                ]

        # ⭐ FALLBACK: If no products match, relax constraints
        if len(filtered) == 0:
            filtered = self._relaxed_filter(constraints)

        return filtered

    def _relaxed_filter(self, constraints: Dict[str, Any]) -> pd.DataFrame:
        """Fallback: relax constraints progressively."""
        # Try removing just caffeine constraint first
        filtered = self.products.copy()

        if constraints.get("category"):
            filtered = filtered[filtered["category"] == constraints["category"]]
        if constraints.get("dairy_free") is True:
            filtered = filtered[filtered["contains_dairy"] == False]
        if constraints.get("max_price") is not None:
            filtered = filtered[filtered["price"] <= constraints["max_price"]]

        if len(filtered) > 0:
            return filtered

        # If still empty, try just category + dietary
        filtered = self.products.copy()
        if constraints.get("category"):
            filtered = filtered[filtered["category"] == constraints["category"]]
        if len(filtered) > 0:
            return filtered

        # Last resort
        return self.products.copy()

In [13]:
# Test filtering
test_constraints = {
    "category": "espresso",
    "temperature": "iced",
    "dairy_free": True,
    "max_calories": None,
    "max_sugar": None,
    "max_price": None,
    "vegan": None,
    "caffeine_level": None
}

product_filter = ProductFilter(products)
filtered = product_filter.filter(test_constraints)
print(f"Filtered to {len(filtered)} products:")
filtered[['product_id', 'name', 'temperature', 'contains_dairy']]

Filtered to 8 products:


,product_id,name,temperature,contains_dairy
15,ICE_001,Iced Caffè Americano,iced,False
21,ICE_007,Iced Brown Sugar Oatmilk Shaken Espresso,iced,False
22,ICE_008,Iced Toasted Vanilla Oatmilk Shaken Espresso,iced,False
23,ICE_009,Iced Chocolate Almondmilk Shaken Espresso,iced,False
29,ICE_015,Iced Hazelnut Oatmilk Shaken Espresso,iced,False
111,ICE_016,Iced Oatmilk Latte,iced,False
112,ICE_017,Iced Coconutmilk Latte,iced,False
113,ICE_018,Iced Almondmilk Latte,iced,False


## Stage 3: Relevance Ranking (Embeddings)

Rank filtered products using semantic similarity with embeddings.

In [14]:
class RelevanceRanker:
    """Stage 3: Rank products by semantic similarity using embeddings."""

    EMBEDDING_MODEL = "models/gemini-embedding-001"

    def __init__(self, products_df: pd.DataFrame):
        self.products = products_df.copy()
        self.product_embeddings = None
        self.product_ids = None

    def _create_product_text(self, row: pd.Series) -> str:
        """Create a rich text representation of a product."""
        text_parts = [
            row["name"],
            row["description"],
            row["category"],
            row["temperature"],
        ]

        # Add simplicity indicators (simpler drinks often rank higher)
        if row["calories"] < 50:
            text_parts.append("light low-calorie simple clean")
        if row["sugar_g"] < 10:
            text_parts.append("unsweetened no sugar")
        if row["is_vegan"]:
            text_parts.append("vegan plant-based")
        if not row["contains_dairy"]:
            text_parts.append("dairy-free no milk")

        # Caffeine descriptors
        caff = row["caffeine_mg"]
        if caff == 0:
            text_parts.append("caffeine-free decaf")
        elif caff <= 50:
            text_parts.append("low caffeine mild")
        elif caff <= 175:
            text_parts.append("medium caffeine regular")
        else:
            text_parts.append("high caffeine strong bold")

        return " | ".join(text_parts)

    def precompute_embeddings(self):
        """Pre-compute embeddings for all products."""
        print("Pre-computing product embeddings...")

        product_texts = []
        self.product_ids = []

        for _, row in self.products.iterrows():
            text = self._create_product_text(row)
            product_texts.append(text)
            self.product_ids.append(row["product_id"])

        # Embed all products in batches
        result = genai.embed_content(
            model=self.EMBEDDING_MODEL,
            content=product_texts,
            task_type="retrieval_document"
        )

        self.product_embeddings = np.array(result['embedding'])
        print(f"Computed embeddings for {len(self.product_ids)} products")
        print(f"Embedding dimension: {self.product_embeddings.shape[1]}")

    def rank(self, query: str, filtered_products: pd.DataFrame) -> List[str]:
        """Rank filtered products by similarity to query."""
        if len(filtered_products) == 0:
            return []

        if len(filtered_products) == 1:
            return filtered_products["product_id"].tolist()

        # Get query embedding
        query_result = genai.embed_content(
            model=self.EMBEDDING_MODEL,
            content=query,
            task_type="retrieval_query"
        )
        query_embedding = np.array(query_result['embedding']).reshape(1, -1)

        # Get embeddings for filtered products
        filtered_ids = filtered_products["product_id"].tolist()
        filtered_indices = [self.product_ids.index(pid) for pid in filtered_ids]
        filtered_embeddings = self.product_embeddings[filtered_indices]

        # Compute cosine similarity
        similarities = cosine_similarity(query_embedding, filtered_embeddings)[0]

        # ⭐ ADD BONUS FOR SIMPLER PRODUCTS (lower calories, lower sugar, lower price)
        for i, pid in enumerate(filtered_ids):
            prod = filtered_products[filtered_products['product_id'] == pid].iloc[0]
            # Small bonus for simpler/basic products
            if prod['calories'] <= 20:
                similarities[i] += 0.02
            if prod['sugar_g'] <= 5:
                similarities[i] += 0.02
            if prod['price'] <= 4.0:
                similarities[i] += 0.01

        # Sort by similarity (descending)
        sorted_indices = np.argsort(similarities)[::-1]
        ranked_products = [filtered_ids[i] for i in sorted_indices]

        return ranked_products

# Initialize the ranker and pre-compute embeddings
ranker = RelevanceRanker(products)
ranker.precompute_embeddings()

Pre-computing product embeddings...
Computed embeddings for 115 products
Embedding dimension: 3072


In [15]:
# Test ranking
test_query = "I want a cold espresso drink without dairy"
test_constraints = {"category": "espresso", "temperature": "iced", "dairy_free": True,
                    "max_calories": None, "max_sugar": None, "max_price": None,
                    "vegan": None, "caffeine_level": None}

product_filter = ProductFilter(products)
filtered = product_filter.filter(test_constraints)
ranked = ranker.rank(test_query, filtered)

print(f"Query: {test_query}")
print(f"\nFiltered to {len(filtered)} products")
print(f"\nRanked results:")
for i, pid in enumerate(ranked[:5]):
    product = products[products['product_id'] == pid].iloc[0]
    print(f"{i+1}. {pid}: {product['name']}")

Query: I want a cold espresso drink without dairy

Filtered to 8 products

Ranked results:
1. ICE_001: Iced Caffè Americano
2. ICE_018: Iced Almondmilk Latte
3. ICE_017: Iced Coconutmilk Latte
4. ICE_009: Iced Chocolate Almondmilk Shaken Espresso
5. ICE_016: Iced Oatmilk Latte


## Complete Pipeline

In [16]:
class StarbucksPipeline:
    """Complete end-to-end recommendation pipeline."""

    def __init__(self, products_df: pd.DataFrame):
        self.extractor = ConstraintExtractor()
        self.filter = ProductFilter(products_df)
        self.ranker = RelevanceRanker(products_df)
        self.ranker.precompute_embeddings()

    def process_query(self, query: str, verbose: bool = False) -> List[str]:
        """Process a single query through all 3 stages."""
        start_time = time.time()

        # Stage 1: Extract constraints
        constraints = self.extractor.extract(query)
        if verbose:
            print(f"Constraints: {constraints}")

        # Stage 2: Filter products
        filtered = self.filter.filter(constraints)
        if verbose:
            print(f"Filtered to {len(filtered)} products")

        # Stage 3: Rank by relevance
        ranked = self.ranker.rank(query, filtered)

        elapsed = time.time() - start_time
        if verbose:
            print(f"Time: {elapsed:.2f}s")

        return ranked

# Initialize the full pipeline
pipeline = StarbucksPipeline(products)

Pre-computing product embeddings...
Computed embeddings for 115 products
Embedding dimension: 3072


In [17]:
# Test the complete pipeline
test_query = "I want something sweet and cold but I'm trying to avoid dairy"
results = pipeline.process_query(test_query, verbose=True)

print(f"\nTop 5 recommendations:")
for i, pid in enumerate(results[:5]):
    product = products[products['product_id'] == pid].iloc[0]
    print(f"{i+1}. {product['name']} (${product['price']}, {product['calories']} cal)")

Constraints: {'category': None, 'temperature': 'iced', 'max_calories': None, 'max_sugar': None, 'max_price': None, 'dairy_free': True, 'vegan': None, 'caffeine_level': None}
Filtered to 34 products
Time: 1.27s

Top 5 recommendations:
1. Iced Passion Tango Tea ($3.25, 0 cal)
2. Iced Green Tea ($3.25, 0 cal)
3. Iced Black Tea ($3.25, 0 cal)
4. Bottled Water ($2.5, 0 cal)
5. Nitro Cold Brew ($5.25, 5 cal)


## Evaluation on Training Data

In [18]:
def calculate_ndcg(predicted: List[str], ground_truth: List[str]) -> float:
    """Calculate NDCG score for a single query."""
    if not ground_truth:
        return 0.0

    k = len(ground_truth)
    relevance_map = {pid: len(ground_truth) - i for i, pid in enumerate(ground_truth)}

    # DCG
    dcg = 0.0
    for i, pid in enumerate(predicted[:k]):
        rel = relevance_map.get(pid, 0)
        dcg += rel / np.log2(i + 2)

    # Ideal DCG
    ideal_relevances = sorted(relevance_map.values(), reverse=True)[:k]
    idcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(ideal_relevances))

    if idcg == 0:
        return 0.0

    return dcg / idcg

In [19]:
# Evaluate on training data (this will take a few minutes)
scores = []

for idx, row in tqdm(train_queries.iterrows(), total=len(train_queries)):
    query_text = row["query_text"]
    ground_truth = ast.literal_eval(row["relevant_products"])

    predicted = pipeline.process_query(query_text)
    score = calculate_ndcg(predicted, ground_truth)
    scores.append(score)

    if score <= 0.8:
      print(f"Query: {query_text}")
      print(f"Predicted: {predicted}")
      print(f"Ground truth: {ground_truth}")
      print(f"NDCG: {score}")

    time.sleep(4)  # Avoid rate limits

avg_ndcg = np.mean(scores)
print(f"\n{'='*60}")
print(f"Average NDCG on Training Data: {avg_ndcg:.4f}")
print(f"{'='*60}")

  0%|          | 0/100 [00:00<?, ?it/s]

Query: need a pick me up, something like just black coffee that's need the caffeine and under 45 grams of sugar
Predicted: ['BRW_003', 'BRW_001', 'BRW_002', 'BRW_005']
Ground truth: ['BRW_002', 'BRW_001', 'BRW_005', 'BRW_003']
NDCG: 0.7857130106485056


  5%|▌         | 5/100 [00:33<10:22,  6.55s/it]

Query: running late but i need a cappuccino or latte that's cold and without milk
Predicted: ['ICE_001', 'ICE_018', 'ICE_017', 'ICE_009', 'ICE_016', 'ICE_015', 'ICE_008', 'ICE_007']
Ground truth: ['ICE_015', 'ICE_008', 'ICE_007', 'ICE_016', 'ICE_001', 'ICE_009', 'ICE_018', 'ICE_017']
NDCG: 0.7588761936604007


  7%|▋         | 7/100 [00:48<11:04,  7.14s/it]WARNING:tornado.access:429 POST /v1beta/models/gemini-2.0-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 7855.26ms


...

Query: lemme get a refresher that's 400 calories or less, vegan friendly, and dairy free
Predicted: ['REF_006', 'REF_009', 'REF_005', 'REF_002', 'REF_008', 'REF_004', 'REF_001', 'REF_003', 'REF_010', 'REF_007']
Ground truth: ['REF_001', 'REF_002', 'REF_008', 'REF_004', 'REF_007', 'REF_005', 'REF_006', 'REF_003', 'REF_009', 'REF_010']
NDCG: 0.7855997593524771


 99%|█████████▉| 99/100 [10:05<00:05,  5.65s/it]

Query: got any basic brewed coffee that's under 40 grams of sugar, less than $4.0, and no regular milk?
Predicted: ['BRW_003', 'BRW_001', 'BRW_004', 'BRW_002']
Ground truth: ['BRW_002', 'BRW_001', 'BRW_004', 'BRW_003']
NDCG: 0.766781143239836


100%|██████████| 100/100 [10:13<00:00,  6.13s/it]


Average NDCG on Training Data: 0.8935


## Generate Test Submission

In [20]:
# Process all test queries
submission_data = []

for idx, row in tqdm(test_queries.iterrows(), total=len(test_queries)):
    query_id = row["query_id"]
    query_text = row["query_text"]

    ranked_products = pipeline.process_query(query_text)
    products_str = ";".join(ranked_products)

    submission_data.append({
        "query_id": query_id,
        "products": products_str
    })

    time.sleep(4)  # Avoid rate limits

submission = pd.DataFrame(submission_data)
print(f"\nGenerated {len(submission)} predictions")

 25%|██▌       | 25/100 [02:17<06:31,  5.22s/it]WARNING:tornado.access:429 POST /v1beta/models/gemini-2.0-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 583.99ms


Error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint: Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.


 27%|██▋       | 27/100 [02:27<06:20,  5.22s/it]WARNING:tornado.access:429 POST /v1beta/models/gemini-2.0-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1116.74ms


Error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint: Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.


 61%|██████    | 61/100 [05:34<03:32,  5.44s/it]WARNING:tornado.access:429 POST /v1beta/models/gemini-2.0-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 913.62ms


Error: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint: Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.


100%|██████████| 100/100 [09:08<00:00,  5.48s/it]


Generated 100 predictions


In [21]:
# Preview submission
submission.head(10)

,query_id,products
0,TEST_001,TEA_002;TEA_010;ICT_002;ICT_009;TEA_011;TEA_00...
1,TEST_002,CBR_009
2,TEST_003,ICT_004;ICT_006;ICT_005;ICT_008;ICT_007
3,TEST_004,BRW_003;BRW_004;BRW_001;BRW_002;BRW_005
4,TEST_005,REF_010
5,TEST_006,CBR_002;CBR_001;CBR_012;CBR_011;CBR_009
6,TEST_007,ICE_001;ICE_018;ICE_012;ICE_013;ICE_007;ICE_01...
7,TEST_008,BRW_002;BRW_003;BRW_001;BRW_004;BRW_005
8,TEST_009,BRW_004;BRW_005;BRW_003;BRW_002;BRW_001
9,TEST_010,BRW_004


In [22]:
# Save submission file
submission.to_csv("submission.csv", index=False)
print("Submission saved to submission.csv")

Submission saved to submission.csv


## Debugging & Analysis Tools

In [ ]:
# Analyze a specific query in detail
def analyze_query(query_text: str, expected_products: List[str] = None):
    """Debug a query through the entire pipeline."""
    print(f"Query: {query_text}")
    print("="*60)

    # Stage 1
    constraints = extractor.extract(query_text)
    print(f"\nStage 1 - Extracted Constraints:")
    print(json.dumps(constraints, indent=2))

    # Stage 2
    filtered = ProductFilter.filter(constraints)
    print(f"\nStage 2 - Filtered Products: {len(filtered)} items")
    if len(filtered) > 0:
        print(filtered[['product_id', 'name', 'category', 'temperature']].head(10))

    # Stage 3
    ranked = ranker.rank(query_text, filtered)
    print(f"\nStage 3 - Ranked Results (top 10):")
    for i, pid in enumerate(ranked[:10]):
        product = products[products['product_id'] == pid].iloc[0]
        match = "✓" if expected_products and pid in expected_products else ""
        print(f"  {i+1}. {pid}: {product['name']} {match}")

    if expected_products:
        print(f"\nExpected products: {expected_products[:10]}")
        ndcg = calculate_ndcg(ranked, expected_products)
        print(f"NDCG Score: {ndcg:.4f}")

# Example usage:
# analyze_query(train_queries.iloc[0]['query_text'],
#               ast.literal_eval(train_queries.iloc[0]['relevant_products']))

In [ ]:
# Analyze a training query
idx = 5  # Change this to analyze different queries
row = train_queries.iloc[idx]
analyze_query(row['query_text'], ast.literal_eval(row['relevant_products']))

Query: running late but i need a cappuccino or latte that's cold and without milk

Stage 1 - Extracted Constraints:
{
  "category": "espresso",
  "temperature": "iced",
  "max_calories": null,
  "max_sugar": null,
  "max_price": null,
  "dairy_free": true,
  "vegan": null,
  "caffeine_level": null
}

Stage 2 - Filtered Products: 8 items
    product_id                                          name  category  \
15     ICE_001                          Iced Caffè Americano  espresso   
21     ICE_007      Iced Brown Sugar Oatmilk Shaken Espresso  espresso   
22     ICE_008  Iced Toasted Vanilla Oatmilk Shaken Espresso  espresso   
23     ICE_009     Iced Chocolate Almondmilk Shaken Espresso  espresso   
29     ICE_015         Iced Hazelnut Oatmilk Shaken Espresso  espresso   
111    ICE_016                            Iced Oatmilk Latte  espresso   
112    ICE_017                        Iced Coconutmilk Latte  espresso   
113    ICE_018                         Iced Almondmilk Latte  espress